In [1]:
# setup + imports
import sys
sys.path.append('..')

import pandas as pd
import re
from pathlib import Path
from config import engine

def extract_uid(filename):
    """Extract uid from filenames like call_log_u01.csv -> u01"""
    match = re.search(r'u\d+', filename)
    return match.group() if match else None

print("✓ Setup complete")


# define path to call_log folder
project_root = Path.cwd().parent
folder = project_root / "dataset" / "archive (1)" / "dataset" / "call_log"

print("Project root:", project_root)
print("Dataset folder:", folder)
print("Folder exists:", folder.exists())


# read all csv files and combine into one dataframe
dfs = []

for file in folder.glob("*.csv"):
    uid = extract_uid(file.name)

    temp_df = pd.read_csv(file)
    temp_df["uid"] = uid

    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print("✓ Raw dataframe created")
print("Shape:", df.shape)
print("DataFrame columns:", df.columns.tolist())
display(df.head())


# optional cleanup: strip spaces from column names
df.columns = df.columns.str.strip()

# check columns that exist in SQL table
table_cols_df = pd.read_sql("""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'call_log'
ORDER BY ordinal_position;
""", engine)

table_cols = table_cols_df["column_name"].tolist()

print("SQL table columns:", table_cols)


# if primary key exists in SQL but not in csv, do not insert it manually
auto_id_cols = {"call_id", "id"}

insert_cols = [col for col in table_cols if col in df.columns and col not in auto_id_cols]

# create dataframe with only columns that match the SQL table
df_to_insert = df[insert_cols].copy()

print("Columns to insert:", df_to_insert.columns.tolist())
print("Shape to insert:", df_to_insert.shape)
display(df_to_insert.head())


# insert into SQL table
df_to_insert.to_sql(
    "call_log",
    engine,
    schema="public",
    if_exists="append",
    index=False
)

print("✓ Data inserted into table 'call_log'")


# verification
check_df = pd.read_sql("SELECT * FROM public.call_log LIMIT 10;", engine)
check_df

✓ Setup complete
Project root: d:\uni\databases\project
Dataset folder: d:\uni\databases\project\dataset\archive (1)\dataset\call_log
Folder exists: True
✓ Raw dataframe created
Shape: (71801, 12)
DataFrame columns: ['id', 'device', 'timestamp', 'CALLS__id', 'CALLS_date', 'CALLS_duration', 'CALLS_name', 'CALLS_number', 'CALLS_numberlabel', 'CALLS_numbertype', 'CALLS_type', 'uid']


,id,device,timestamp,CALLS__id,CALLS_date,CALLS_duration,CALLS_name,CALLS_number,CALLS_numberlabel,CALLS_numbertype,CALLS_type,uid
0,8b67cbb5-cfa2-465b-b5af-31ee2c6606a6-40,1977b545-a88f-4903-a7ae-2c434de4be49,1364099483,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00
1,9c0354e8-4f4f-451a-9358-efe17df0d26c-22,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,213.0,1.364077e+12,10.0,"{""ONE_WAY_HASH"":""70f8bb9a8a5393ef080507a89e4b9...","{""ONE_WAY_HASH"":""c51aad31359925622a961f2082ecf...",NaN,"{""ONE_WAY_HASH"":""356a192b7913b04c54574d18c28d4...",2.0,u00
2,9c0354e8-4f4f-451a-9358-efe17df0d26c-22,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,212.0,1.364077e+12,13.0,"{""ONE_WAY_HASH"":""70f8bb9a8a5393ef080507a89e4b9...","{""ONE_WAY_HASH"":""c51aad31359925622a961f2082ecf...",NaN,"{""ONE_WAY_HASH"":""356a192b7913b04c54574d18c28d4...",2.0,u00
3,9c0354e8-4f4f-451a-9358-efe17df0d26c-22,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,211.0,1.364075e+12,18.0,"{""ONE_WAY_HASH"":""70f8bb9a8a5393ef080507a89e4b9...","{""ONE_WAY_HASH"":""c51aad31359925622a961f2082ecf...",NaN,"{""ONE_WAY_HASH"":""356a192b7913b04c54574d18c28d4...",2.0,u00
4,9c0354e8-4f4f-451a-9358-efe17df0d26c-22,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,210.0,1.364074e+12,438.0,"{""ONE_WAY_HASH"":""70f8bb9a8a5393ef080507a89e4b9...","{""ONE_WAY_HASH"":""c51aad31359925622a961f2082ecf...",NaN,"{""ONE_WAY_HASH"":""356a192b7913b04c54574d18c28d4...",2.0,u00


SQL table columns: ['call_id', 'uid', 'record_id', 'device', 'timestamp', 'calls__id', 'calls_date', 'calls_duration', 'calls_name', 'calls_number', 'calls_numberlabel', 'calls_numbertype', 'calls_type']
Columns to insert: ['uid', 'device', 'timestamp']
Shape to insert: (71801, 3)


,uid,device,timestamp
0,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364099483
1,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455
2,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455
3,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455
4,u00,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455


✓ Data inserted into table 'call_log'


,call_id,uid,record_id,device,timestamp,calls__id,calls_date,calls_duration,calls_name,calls_number,calls_numberlabel,calls_numbertype,calls_type
0,1,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364099483,None,None,None,None,None,None,None,None
1,2,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
2,3,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
3,4,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
4,5,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
5,6,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
6,7,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
7,8,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
8,9,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
9,10,u00,None,1977b545-a88f-4903-a7ae-2c434de4be49,1364077455,None,None,None,None,None,None,None,None
